# 4. Validação

Avaliação do modelo treinado sobre o split de teste. Caso não exista um run de treino com o `EXPERIMENT_ID` informado, o modelo base é avaliado.

## Preparação

In [8]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from src import config, datasets, eval as eval_mod

## Configuração

In [ ]:
EXPERIMENT_ID        = "yolo-direct-all-datasets"  # identifica os pesos do treino
N_SAMPLE_PREDICTIONS = 10
DATASET_STAGE        = "processed"   # "interim" or "processed"
WEIGHTS              = ""  # deixar vazio para carregar automaticamente do run de treino

# datasets de avaliação — independentes dos usados no treino
EVAL_DATASETS = datasets.available(stage=DATASET_STAGE)
# opções: ["inside-bus-view", "passenger-deakin", "passenger-detection-bus", "crowdhuman"]

## Datasets

In [10]:
data_spec = datasets.prepare(EVAL_DATASETS, stage=DATASET_STAGE)

for name in EVAL_DATASETS:
    for split in ["valid", "test"]:
        img_dir = config.DATA_DIR / DATASET_STAGE / name / split / "images"
        lbl_dir = config.DATA_DIR / DATASET_STAGE / name / split / "labels"
        if img_dir.exists():
            n_img = len(list(img_dir.glob("*.*")))
            n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
            print(f"  {name}/{split}: {n_img} imagens | {n_lbl} labels")

  crowdhuman/valid: 200 imagens | 200 labels
  crowdhuman/test: 200 imagens | 200 labels
  inside-bus-view/valid: 13 imagens | 13 labels
  inside-bus-view/test: 15 imagens | 15 labels
  passenger-deakin/valid: 96 imagens | 96 labels
  passenger-deakin/test: 97 imagens | 97 labels


## Pesos

In [5]:
run_dir, weights_path = eval_mod.find_run(EXPERIMENT_ID)
weights_path = WEIGHTS or weights_path

print("Run dir:", run_dir)
print("Pesos:  ", weights_path)

Run dir: /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-direct-all-datasets_20260617_170957
Pesos:   /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-direct-all-datasets_20260617_170957/train/weights/best.pt


## Modelo

In [6]:
from ultralytics import YOLO
model = YOLO(str(weights_path))

## Avaliação

In [7]:
metrics = eval_mod.run(EXPERIMENT_ID, model, data_spec,
                       n_samples=N_SAMPLE_PREDICTIONS, run_dir=run_dir)
metrics

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ruben/.netrc.
wandb: Currently logged in as: vinicius_trentin (IA901-2026S1-bus-passenger-count) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: run 'yolo-direct-all-datasets_20260617_170957_eval' -> https://wandb.ai/IA901-2026S1-bus-passenger-count/bus-passenger-count/runs/zxb8i2ze
Ultralytics 8.4.50 🚀 Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


[W622 19:25:54.343658313 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.344873934 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.345590819 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.345940257 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.346113196 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.346312151 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.346497286 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.346665429 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.346891641 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:25:54.347176785 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W622 19:2

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2152.9±1097.7 MB/s, size: 294.6 KB)
val: Scanning /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/processed/crowdhuman/test/labels... 312 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 312/312 1.6Kit/s 0.2s<0.3s
val: New cache created: /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/processed/crowdhuman/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 5.6it/s 3.6s0.1s
                   all        312       5054       0.35       0.28      0.222     0.0937
Speed: 0.8ms preprocess, 6.4ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-direct-all-datasets_20260617_170957/eval
avaliação(eval): métricas:
 test/metrics/precision(B)                0.3500
 test/metrics/recall(B)                   0.2797
 test/metrics/mAP50(B)     

wandb: registrado 10 imagens em 'test/predictions'


final/test/fitness,▁
final/test/metrics/mAP50(B),▁
final/test/metrics/mAP50-95(B),▁
final/test/metrics/precision(B),▁
final/test/metrics/recall(B),▁
test/fitness,▁
test/metrics/mAP50(B),▁
test/metrics/mAP50-95(B),▁
test/metrics/precision(B),▁
test/metrics/recall(B),▁
final/test/fitness,0.09371


wandb: execução(run) finalizada


{'test/metrics/precision(B)': 0.3500097541098913,
 'test/metrics/recall(B)': 0.27968385508731974,
 'test/metrics/mAP50(B)': 0.22236280293767136,
 'test/metrics/mAP50-95(B)': 0.09371378336268824,
 'test/fitness': 0.09371378336268824}